In [2]:
import os
print(os.getcwd())
os.chdir('/home/fatemeh/thesis/kinodata-3D-affinity-prediction')
print(os.getcwd())

/home/fatemeh/thesis/kinodata-3D-affinity-prediction
/home/fatemeh/thesis/kinodata-3D-affinity-prediction


In [4]:
# import kinodata-3d-affinity-prediction
import kinodata
from kinodata.data import KinodataDocked
from kinodata.transform import TransformToComplexGraph
from kinodata.types import *


import json
from pathlib import Path
from typing import Any
from functools import partial

import torch

import kinodata.configuration as cfg
# from kinodata.model import ComplexTransformer, DTIModel, RegressionModel
from kinodata.model import ComplexTransformer, RegressionModel
from kinodata.model.complex_transformer import make_model as make_complex_transformer
# from kinodata.model.dti import make_model as make_dti_baseline
from kinodata.data.data_module import make_kinodata_module
from kinodata.transform import TransformToComplexGraph

import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import tqdm


!wandb disabled

W&B disabled.


## Whole Data
Saving a portion

In [3]:
data = KinodataDocked(transform=TransformToComplexGraph(remove_heterogeneous_representation=False))

In [4]:
mini_data = data[:100]
mini_data
torch.save(mini_data, "100subset_data.pt")

NameError: name 'data' is not defined

# Load mini

In [ ]:
mini_data = torch.load("/data/probing/100subset_data.pt", weights_only=False)

In [6]:
from torch_geometric.loader import DataLoader

mini_loader = DataLoader(
        mini_data,
        batch_size=20,
    )

Data module, but crashes

In [5]:
data_module = make_kinodata_module(
    cfg.get("data", "training").update(
        dict(
            batch_size=32,
            split_type="scaffold-k-fold",
            filter_rmsd_max_value=2.0,
            split_index=0,
        )
    ),
    transforms=[TransformToComplexGraph(remove_heterogeneous_representation=False)],
)

: 

# Looking at data

In [41]:
# print(data)
# print(dir(data))
print(data.__dict__)
# print(data[0])

{'remove_hydrogen': True, '_prefix': None, 'residue_representation': 'sequence', 'require_kissim_residues': False, 'use_multiprocessing': True, 'make_pyg_data': functools.partial(<function process_pyg at 0x7fb3ace4ede0>, residue_representation='sequence', require_kissim_residues=False), 'num_processes': 16, 'post_filter': None, 'root': '/mnt/d/Work/Thesis/kinosachen/kinodata-3D-affinity-prediction/data', 'transform': None, 'pre_transform': None, 'pre_filter': FilterActivityType(allowed=pIC50), 'log': True, '_indices': None, 'force_reload': False, '_data': HeteroData(
  y=[41238],
  docking_score=[41238],
  posit_prob=[41238],
  predicted_rmsd=[41238],
  pocket_sequence=[41238],
  scaffold=[41238],
  activity_type=[41238],
  ident=[41238],
  smiles=[41238],
  ligand={
    z=[1288419],
    x=[1288419, 12],
    pos=[1288419, 3],
  },
  pocket={
    z=[27393111],
    x=[27393111, 12],
    pos=[27393111, 3],
  },
  pocket_residue={ x=[3498751, 23] },
  (ligand, bond, ligand)={
    edge_inde

In [42]:
# print(data.{'pocket'})
# print(data.data.keys())
# print(data.data.edge_types)
# print(data.data.node_types)
# print(data[2].edge_attrs)
# print(data[2].node_attrs)
# print(data[2].edge_stores)
print(data[2])

['posit_prob', 'pos', 'scaffold', 'ident', 'y', 'edge_attr', 'pocket_sequence', 'activity_type', 'edge_index', 'docking_score', 'predicted_rmsd', 'z', 'x', 'smiles']


In [10]:
data.data

/home/fatemeh/miniconda3/envs/kinodata3d/lib/python3.12/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache now by removing all elements in `dataset._data_list`. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


HeteroData(
  y=[41238],
  docking_score=[41238],
  posit_prob=[41238],
  predicted_rmsd=[41238],
  pocket_sequence=[41238],
  scaffold=[41238],
  activity_type=[41238],
  ident=[41238],
  smiles=[41238],
  ligand={
    z=[1288419],
    x=[1288419, 12],
    pos=[1288419, 3],
  },
  pocket={
    z=[27393111],
    x=[27393111, 12],
    pos=[27393111, 3],
  },
  pocket_residue={ x=[3498751, 23] },
  (ligand, bond, ligand)={
    edge_index=[2, 2857602],
    edge_attr=[2857602, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 55219856],
    edge_attr=[55219856, 4],
  }
)

In [7]:
sample = data[1]
node_types, edge_types = sample.metadata()
print(f"Node types: {', '.join([nt for nt in node_types])}")
print(f"Edge types: {', '.join([str(et) for et in edge_types])}")
print(f"Number of ligand heavy atoms: {sample['ligand'].x.size(0)}")
print(f"Number of pocket heavy atoms: {sample['pocket'].x.size(0)}")
print(f"Position of ligand atom indexed 0: {sample['ligand'].pos[0]}")
print(f"Position of pocket atom indexed 0: {sample['pocket'].pos[0]}")

Node types: ligand, pocket, pocket_residue
Edge types: ('ligand', 'bond', 'ligand'), ('pocket', 'bond', 'pocket')
Number of ligand heavy atoms: 26
Number of pocket heavy atoms: 652
Position of ligand atom indexed 0: tensor([ 5.4872, 25.3067, 37.4988])
Position of pocket atom indexed 0: tensor([ 9.5601, 17.7450, 49.1304])


In [9]:
graf = TransformToComplexGraph()(sample)
graf

HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA',
  scaffold='C1CCC(CCC2CCC(CC3CCCCC3)C3CCCCC23)CC1',
  activity_type='pIC50',
  ident=[1],
  smiles='Clc1ccc(Nc2nnc(NCc3ccncc3)c3ccccc23)cc1',
  ligand={
    z=[26],
    x=[26, 12],
    pos=[26, 3],
  },
  pocket={
    z=[652],
    x=[652, 12],
    pos=[652, 3],
  },
  pocket_residue={ x=[85, 23] },
  complex={
    x=[678, 12],
    z=[678],
    pos=[678, 3],
  },
  (ligand, bond, ligand)={
    edge_index=[2, 58],
    edge_attr=[58, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 1308],
    edge_attr=[1308, 4],
  },
  (complex, bond, complex)={
    edge_index=[2, 1366],
    edge_attr=[1366, 4],
  }
)

In [25]:
import networkx as nx
from pyvis.network import Network

def interactive_hetero_graph(data):
    # Initialize PyVis network object with remote CDN resources to avoid permission issues
    net = Network(notebook=True, width="1000px", height="700px", directed=False, cdn_resources="remote")

    # Loop over each node type in HeteroData
    for node_type in data.node_types:
        node_indices = data[node_type].num_nodes
        for i in range(node_indices):
            node_id = f"{node_type}_{i}"
            # Color nodes differently based on their type
            color = 'blue' if node_type == 'protein' else 'green' if node_type == 'ligand' else 'gray'
            net.add_node(node_id, label=node_id, color=color)

    # Loop over each edge type in HeteroData
    for edge_type in data.edge_types:
        edge_index = data[edge_type].edge_index
        source_node_type, relation_type, target_node_type = edge_type

        # Adding edges to PyVis graph
        for i in range(edge_index.size(1)):  # edge_index is (2, num_edges)
            source = edge_index[0, i].item()
            target = edge_index[1, i].item()

            source_id = f"{source_node_type}_{source}"
            target_id = f"{target_node_type}_{target}"

            net.add_edge(source_id, target_id, label=relation_type)

    # Display the interactive graph
    return net.show("hetero_graph.html")

# Usage: assuming your HeteroData object is loaded as 'data'
interactive_hetero_graph(sample)


hetero_graph.html


##

# Loading the model

In [7]:
print(Path.cwd())

/home/fatemeh/thesis/kinodata-3D-affinity-prediction


In [7]:
model_dir = Path("models")
# print(model_dir)
assert model_dir.exists()

def path_to_model(rmsd_threshold: int, split_type: str, split_fold: int, model_type: str) -> Path:
    p = model_dir / f"rmsd_cutoff_{rmsd_threshold}" / split_type / str(split_fold) / model_type
    if not p.exists():
        p.mkdir(parents=True)
    return p

cgnn_3d_path = path_to_model(rmsd_threshold=2, split_type="scaffold-k-fold", split_fold=0, model_type="CGNN-3D")
cgnn_3d_ckpt = list(cgnn_3d_path.glob("**/*.ckpt"))[0]
# print(model_ckpt)
cgnn_3d_config = cgnn_3d_path / "config.json"
# print(model_config)

In [8]:
def load_wandb_config(
    config_file: Path
) -> dict[str, Any]:
    with open(config_file, "r") as f_config:
        config = json.load(f_config)
    config = {str(key): value["value"] for key, value in config.items()}
    return config

In [9]:
config = cfg.Config(load_wandb_config(cgnn_3d_config))
cgnn_3d = make_complex_transformer(config)

In [10]:
model_cls = {
    # "DTI": make_dti_baseline,
    # "CGNN": make_complex_transformer,
    "CGNN-3D": make_complex_transformer
}

def load_from_checkpoint(model: RegressionModel, model_ckpt: str) -> RegressionModel:
    ckp = torch.load(model_ckpt, map_location="cpu")
    assert isinstance(model, RegressionModel)
    model.load_state_dict(ckp["state_dict"])
    return model

In [11]:
cgnn_3d_loaded = load_from_checkpoint(model = cgnn_3d, model_ckpt=cgnn_3d_ckpt)
cgnn_3d_loaded.train(False)

ComplexTransformer(
  (criterion): MSELoss()
  (act): SiLU()
  (interaction_module): CombinedInteractions(
    (interactions): ModuleList(
      (0): CovalentInteractions(
        (act): SiLU()
        (lin): Linear(in_features=4, out_features=256, bias=True)
      )
      (1): StructuralInteractions(
        (act): SiLU()
        (distance_embedding): GaussianDistEmbedding()
        (lin): Linear(in_features=256, out_features=256, bias=False)
      )
    )
    (act): SiLU()
  )
  (atomic_num_embedding): Embedding(100, 256)
  (lin_atom_features): Linear(in_features=12, out_features=256, bias=True)
  (attention_blocks): ModuleList(
    (0-2): 3 x SPAB(
      (attention): SparseAttention(
        (lin_query): Linear(in_features=256, out_features=256, bias=False)
        (lin_key_value): Linear(in_features=256, out_features=512, bias=False)
        (lin_bias): Linear(in_features=256, out_features=512, bias=False)
        (lin_out): Linear(in_features=256, out_features=256, bias=False)
   

In [12]:
config

Config(lr=0.0001, act=silu, ln1=True, ln2=True, ln3=True, seed=420, optim=adamw, epochs=300, k_fold=5, min_lr=1e-06, dropout=0, dry_run=False, loss_type=mse, lr_factor=0.9, num_heads=4, use_bonds=True, batch_size=42, data_split=None, edge_types=[['ligand', 'intraacts', 'ligand'], ['ligand', 'interacts', 'pocket'], ['pocket', 'interacts', 'ligand']], graph_norm=False, node_types=['complex'], split_type=scaffold-k-fold, accelerator=gpu, lr_patience=8, num_workers=32, split_index=0, weight_decay=3e-06, edge_attr_size=4, need_distances=False, clip_grad_value=None, hidden_channels=256, remove_hydrogen=True, interaction_modes=['covalent', 'structural'], max_num_neighbors=16, add_docking_scores=False, interaction_radius=6, num_attention_blocks=3, num_residue_features=6, add_artificial_decoys=False, filter_rmsd_max_value=2, accumulate_grad_batches=3, early_stopping_patience=24, additional_atom_features=False, perturb_ligand_positions=0, perturb_pocket_positions=0, perturb_complex_positions=0.1

In [13]:
import numpy as np
from kinodata.model.regression import RegressionModel, cat_many
from torch_geometric.loader import DataLoader
from pytorch_lightning import Trainer
import pandas as pd


def predict_df(
    model: RegressionModel,
    loader: DataLoader,
    trainer: Trainer | None = None,
    ckpt_path: str | None = "best",
) -> pd.DataFrame:
    if trainer is None:
        trainer = Trainer()
    dict_list = trainer.predict(model, loader, ckpt_path=ckpt_path)
    return pd.DataFrame(
        {key: np.array(value) for key, value in cat_many(dict_list).items()}
    )

In [14]:
# from kinodata.training.predict import predict_df
df = predict_df(cgnn_3d, mini_loader, ckpt_path = cgnn_3d_ckpt)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/fatemeh/miniconda3/envs/kinodata3d/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
Restoring states from the checkpoint path at models/rmsd_cutoff_2/scaffold-k-fold/0/CGNN-3D/model.ckpt
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v2.2.2. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint models/

Predicting DataLoader 0: 100%|██████████| 5/5 [00:36<00:00,  0.14it/s]


In [ ]:
df.to_pickle("/data/probing/y_prob_exp_1.pkl")

# Hook

In [42]:
from kinodata.model.complex_transformer import ComplexTransformer
from torch import Tensor
from typing import Tuple, Dict
from torch_geometric.data import HeteroData

class ProbingModel(ComplexTransformer):
    def __init__(self, original_model, **kwargs) -> None:
        super().__init__(**kwargs)
        self.original_model = original_model
        self.hidden_states: Dict[str, Tuple[str, Tensor]] = {}

    def forward(self, data: HeteroData) -> Tensor:
        node_store = data[NodeType.Complex]
        node_repr = self.initial_embed_nodes(data)
        edge_index, edge_repr = self.initial_embed_edges(data)

        # Clear previous hidden states
        self.hidden_states.clear()
        
        # Capture the initial representations
        self.hidden_states['initial_node_repr'] = ('initial_node_repr', node_repr.clone())
        self.hidden_states['initial_edge_repr'] = ('initial_edge_repr', edge_repr.clone())
        
        for i, (sparse_attention_block, norm) in enumerate(zip(self.attention_blocks, self.norm_layers)):
            # Run the sparse attention block
            node_repr, edge_repr = sparse_attention_block(node_repr, edge_repr, edge_index)

            # Capture node representation after attention block
            self.hidden_states[f'node_repr_after_block_{i}'] = ('node_repr_after_block', node_repr.clone())
            self.hidden_states[f'edge_repr_after_block_{i}'] = ('edge_repr_after_block', edge_repr.clone())
            
            # Apply normalization
            node_repr = norm(node_repr, node_store.batch)

            # Capture node representation after normalization
            self.hidden_states[f'node_repr_after_norm_{i}'] = ('node_repr_after_norm', node_repr.clone())

        # Aggregate the final node representations
        graph_repr = self.aggr(node_repr, node_store.batch)

        # Capture the final graph representation
        self.hidden_states['final_graph_repr'] = ('final_graph_repr', graph_repr.clone())
        
        return self.out(graph_repr)

In [ ]:
def make_probing_model(model: ComplexTransformer, config: cfg.Config) -> ProbingModel:
    cls = partial(model, config)
    return config.init(cls)